# 批量结构分析工作流

**主要功能：**
- 批量分析多个蛋白质结构
- 结构质量评估和比较
- 结构相似性分析
- 生成综合分析报告

**输入：**
- 多个PDB结构文件
- 或包含PDB文件的目录

**输出：**
- 结构质量报告
- 结构相似性矩阵
- 聚类分析结果
- 可视化图表

**系统要求：**
- 已安装结构分析工具
- 约 5-10 GB 磁盘空间
- 适合大规模结构数据集

## 1. 环境设置与依赖安装

In [ ]:
# 环境设置与依赖检查
import sys
from pathlib import Path

# Add protflow to path
project_root = Path.cwd()
while not (project_root / 'src' / 'protflow').exists() and project_root != project_root.parent:
    project_root = project_root.parent

if (project_root / 'src').exists():
    src_dir = str(project_root / 'src')
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    print(f"✓ protflow 路径: {src_dir}")

# Setup environment (automatically checks and installs core dependencies)
from protflow.utils.notebook_utils import (
    setup_analysis_notebook,
    check_and_install_packages
)

paths = setup_analysis_notebook(work_dir_name='batch_analysis_runs')

# Common imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

# Check additional analysis packages
print("\n检查额外分析包...")
check_and_install_packages([
    'seaborn',
    'scipy',
    ('sklearn', 'scikit-learn'),
    'plotly'
])

# Store paths
PROJECT_ROOT = paths['PROJECT_ROOT']
WORK_DIR = paths['WORK_DIR']
DATA_DIR = paths['DATA_DIR']

print(f"\n✓ 初始化完成. 工作目录: {WORK_DIR}")

## 2. 输入结构文件收集

In [ ]:
from pathlib import Path
import re

def collect_structure_files(input_path):
    """收集结构文件"""
    input_path = Path(input_path)
    structure_files = []
    
    if input_path.is_file():
        # 单个文件
        if input_path.suffix.lower() == '.pdb':
            structure_files.append(input_path)
            print(f"✓ 单个文件: {input_path.name}")
        else:
            print(f"❌ 不支持的文件格式: {input_path.suffix}")
    
    elif input_path.is_dir():
        # 目录
        pdb_files = list(input_path.glob('*.pdb'))
        pdb_files.extend(input_path.glob('*.PDB'))
        
        # 递归搜索
        if len(pdb_files) < 10:  # 如果直接搜索找到的文件较少
            pdb_files.extend(input_path.rglob('*.pdb'))
            pdb_files.extend(input_path.rglob('*.PDB'))
        
        # 去重并排序
        pdb_files = sorted(list(set(pdb_files)), key=lambda x: x.name)
        structure_files = pdb_files
        
        print(f"✓ 在目录中找到 {len(structure_files)} 个PDB文件")
    
    else:
        print(f"❌ 路径不存在: {input_path}")
    
    # 验证文件
    valid_files = []
    for pdb_file in structure_files:
        if pdb_file.exists() and pdb_file.stat().st_size > 0:
            valid_files.append(pdb_file)
        else:
            print(f"⚠️ 跳过无效文件: {pdb_file}")
    
    # 提取蛋白质名称
    protein_info = []
    for pdb_file in valid_files:
        # 从文件名提取蛋白质名称
        protein_name = pdb_file.stem
        
        # 清理名称 (移除特殊字符)
        protein_name = re.sub(r'[^\w\-_.]', '_', protein_name)
        
        protein_info.append({
            'file_path': str(pdb_file),
            'file_name': pdb_file.name,
            'protein_name': protein_name,
            'file_size': pdb_file.stat().st_size
        })
    
    print(f"✓ 有效文件: {len(protein_info)} 个")
    
    # 显示前10个文件
    if len(protein_info) > 0:
        print("\n前10个文件:")
        for i, info in enumerate(protein_info[:10], 1):
            size_mb = info['file_size'] / (1024 * 1024)
            print(f"  {i:2d}. {info['file_name']} ({size_mb:.1f} MB)")
        
        if len(protein_info) > 10:
            print(f"      ... 还有 {len(protein_info) - 10} 个文件")
    
    return protein_info

# 示例输入路径
input_structures = "path/to/pdb/files"  # 替换为您的路径
# input_structures = "path/to/structure.pdb"  # 单个文件

structure_info = collect_structure_files(input_structures)

## 3. 结构质量评估

In [ ]:
from Bio.PDB import PDBParser, PDBIO
from Bio.PDB.PDBExceptions import PDBConstructionException

def assess_structure_quality(pdb_file):
    """评估单个结构的质量"""
    
    quality_metrics = {
        'file_path': str(pdb_file),
        'file_name': Path(pdb_file).name,
        'status': 'failed',
        'num_models': 0,
        'num_chains': 0,
        'num_residues': 0,
        'num_atoms': 0,
        'residue_types': set(),
        'missing_atoms': 0,
        'missing_residues': 0,
        'sequence_gaps': 0,
        'clashes': 0,
        'ramachandran_outliers': 0,
        'error_message': None
    }
    
    try:
        parser = PDBParser(PERMISSIVE=1, QUIET=1)
        structure = parser.get_structure('temp', pdb_file)
        
        quality_metrics['status'] = 'success'
        quality_metrics['num_models'] = len(structure)
        
        # 统计链、残基、原子数量
        chains = []
        residues = []
        atoms = []
        
        for model in structure:
            for chain in model:
                chains.append(chain.id)
                for residue in chain:
                    residues.append(residue.id[1])  # 残基序号
                    quality_metrics['residue_types'].add(residue.resname)
                    for atom in residue:
                        atoms.append(atom.name)
        
        quality_metrics['num_chains'] = len(set(chains))
        quality_metrics['num_residues'] = len(residues)
        quality_metrics['num_atoms'] = len(atoms)
        
        # 简单的质量检查
        if quality_metrics['num_residues'] < 10:
            quality_metrics['status'] = 'poor'
            quality_metrics['error_message'] = '残基数过少'
        elif quality_metrics['num_atoms'] < quality_metrics['num_residues'] * 5:
            quality_metrics['status'] = 'incomplete'
            quality_metrics['error_message'] = '原子数不足，可能结构不完整'
        
    except PDBConstructionException as e:
        quality_metrics['error_message'] = f"PDB格式错误: {str(e)}"
    
    except Exception as e:
        quality_metrics['error_message'] = f"解析错误: {str(e)}"
    
    # 转换集合为列表 (为了JSON序列化)
    quality_metrics['residue_types'] = list(quality_metrics['residue_types'])
    
    return quality_metrics

def batch_quality_assessment(structure_info):
    """批量质量评估"""
    
    print("=== 结构质量评估 ===")
    print(f"待评估结构: {len(structure_info)} 个")
    
    quality_results = []
    
    for info in tqdm(structure_info, desc="质量评估"):
        quality = assess_structure_quality(info['file_path'])
        quality.update(info)  # 合并信息
        quality_results.append(quality)
    
    # 统计结果
    status_counts = {}
    total_residues = 0
    total_atoms = 0
    successful_structures = 0
    
    for result in quality_results:
        status = result['status']
        status_counts[status] = status_counts.get(status, 0) + 1
        
        if result['status'] == 'success':
            successful_structures += 1
            total_residues += result['num_residues']
            total_atoms += result['num_atoms']
    
    print(f"\n=== 质量评估结果 ===")
    print(f"总结构数: {len(quality_results)}")
    
    for status, count in status_counts.items():
        percentage = (count / len(quality_results)) * 100
        print(f"  {status}: {count} ({percentage:.1f}%)")
    
    if successful_structures > 0:
        print(f"\n成功结构统计:")
        print(f"  平均残基数: {total_residues / successful_structures:.1f}")
        print(f"  平均原子数: {total_atoms / successful_structures:.1f}")
    
    # 保存结果
    results_df = pd.DataFrame(quality_results)
    quality_file = WORK_DIR / 'structure_quality.csv'
    results_df.to_csv(quality_file, index=False)
    print(f"\n✓ 质量评估结果保存: {quality_file}")
    
    return quality_results

# 运行批量质量评估
if structure_info:
    quality_results = batch_quality_assessment(structure_info)
else:
    quality_results = []
    print("⚠️ 没有结构文件可供评估")

## 4. 结构相似性分析

In [ ]:
def calculate_structure_similarity(pdb_file1, pdb_file2):
    """计算两个结构之间的相似性"""
    
    try:
        from Bio.PDB import PDBParser, Superimposer
        import numpy as np
        
        parser = PDBParser(PERMISSIVE=1, QUIET=1)
        
        # 解析结构
        structure1 = parser.get_structure('s1', pdb_file1)
        structure2 = parser.get_structure('s2', pdb_file2)
        
        # 提取Cα原子坐标
        atoms1 = []
        atoms2 = []
        
        for model in structure1:
            for chain in model:
                for residue in chain:
                    if residue.has_id('CA'):
                        atoms1.append(residue['CA'])
        
        for model in structure2:
            for chain in model:
                for residue in chain:
                    if residue.has_id('CA'):
                        atoms2.append(residue['CA'])
        
        if len(atoms1) == 0 or len(atoms2) == 0:
            return {
                'rmsd': None,
                'similarity_score': 0,
                'common_residues': 0,
                'length_ratio': 0,
                'status': 'no_common_atoms'
            }
        
        # 使用较短的序列长度
        min_length = min(len(atoms1), len(atoms2))
        atoms1 = atoms1[:min_length]
        atoms2 = atoms2[:min_length]
        
        # 计算RMSD
        sup = Superimposer()
        sup.set_atoms(atoms1, atoms2)
        rmsd = sup.rms
        
        # 计算相似性分数 (基于RMSD)
        # RMSD < 2.0 Å: 高相似性
        # RMSD 2.0-4.0 Å: 中等相似性
        # RMSD > 4.0 Å: 低相似性
        if rmsd is not None:
            if rmsd < 2.0:
                similarity_score = 1.0 - (rmsd / 2.0) * 0.5
            elif rmsd < 4.0:
                similarity_score = 0.5 - ((rmsd - 2.0) / 2.0) * 0.4
            else:
                similarity_score = 0.1 - min((rmsd - 4.0) / 6.0, 0.09)
            
            similarity_score = max(0.01, similarity_score)  # 最小相似度0.01
        else:
            similarity_score = 0
        
        return {
            'rmsd': rmsd,
            'similarity_score': similarity_score,
            'common_residues': min_length,
            'length_ratio': min_length / max(len(atoms1), len(atoms2)),
            'status': 'success'
        }
    
    except Exception as e:
        return {
            'rmsd': None,
            'similarity_score': 0,
            'common_residues': 0,
            'length_ratio': 0,
            'status': f'error: {str(e)}'
        }

def batch_similarity_analysis(quality_results):
    """批量相似性分析"""
    
    # 筛选成功的结构
    successful_structures = [
        r for r in quality_results 
        if r['status'] == 'success' and r['num_residues'] >= 30
    ]
    
    if len(successful_structures) < 2:
        print(f"⚠️ 需要至少2个有效结构进行分析，当前有 {len(successful_structures)} 个")
        return None
    
    print(f"=== 结构相似性分析 ===")
    print(f"待分析结构: {len(successful_structures)} 个")
    
    # 创建相似性矩阵
    n_structures = len(successful_structures)
    similarity_matrix = np.zeros((n_structures, n_structures))
    rmsd_matrix = np.full((n_structures, n_structures), np.nan)
    
    similarity_results = []
    
    # 计算所有结构对的相似性
    with tqdm(total=n_structures * (n_structures - 1) // 2, desc="相似性计算") as pbar:
        for i in range(n_structures):
            for j in range(i + 1, n_structures):
                
                result = calculate_structure_similarity(
                    successful_structures[i]['file_path'],
                    successful_structures[j]['file_path']
                )
                
                # 填充矩阵 (对称)
                similarity_matrix[i, j] = result['similarity_score']
                similarity_matrix[j, i] = result['similarity_score']
                
                if result['rmsd'] is not None:
                    rmsd_matrix[i, j] = result['rmsd']
                    rmsd_matrix[j, i] = result['rmsd']
                
                # 保存详细结果
                similarity_results.append({
                    'structure1': successful_structures[i]['protein_name'],
                    'structure2': successful_structures[j]['protein_name'],
                    'rmsd': result['rmsd'],
                    'similarity_score': result['similarity_score'],
                    'common_residues': result['common_residues'],
                    'status': result['status']
                })
                
                pbar.update(1)
    
    # 对角线设为1 (自相似)
    np.fill_diagonal(similarity_matrix, 1.0)
    np.fill_diagonal(rmsd_matrix, 0.0)
    
    print(f"✓ 相似性分析完成")
    
    # 统计结果
    valid_similarities = [r['similarity_score'] for r in similarity_results if r['status'] == 'success']
    valid_rmsds = [r['rmsd'] for r in similarity_results if r['rmsd'] is not None]
    
    if valid_similarities:
        print(f"相似性统计:")
        print(f"  平均相似性: {np.mean(valid_similarities):.3f}")
        print(f"  相似性范围: {np.min(valid_similarities):.3f} - {np.max(valid_similarities):.3f}")
        
        # 高相似性结构对
        high_similarity = [r for r in similarity_results if r['similarity_score'] > 0.8]
        print(f"  高相似性对 (>0.8): {len(high_similarity)}")
        
        if high_similarity:
            print(f"  最相似的结构对:")
            best_pair = max(high_similarity, key=lambda x: x['similarity_score'])
            print(f"    {best_pair['structure1']} vs {best_pair['structure2']}")
            print(f"    RMSD: {best_pair['rmsd']:.2f} Å, 相似性: {best_pair['similarity_score']:.3f}")
    
    if valid_rmsds:
        print(f"RMSD统计:")
        print(f"  平均RMSD: {np.mean(valid_rmsds):.2f} Å")
        print(f"  RMSD范围: {np.min(valid_rmsds):.2f} - {np.max(valid_rmsds):.2f} Å")
    
    # 保存结果
    results_df = pd.DataFrame(similarity_results)
    similarity_file = WORK_DIR / 'structure_similarity.csv'
    results_df.to_csv(similarity_file, index=False)
    print(f"\n✓ 相似性结果保存: {similarity_file}")
    
    # 保存矩阵
    structure_names = [s['protein_name'] for s in successful_structures]
    similarity_matrix_df = pd.DataFrame(similarity_matrix, 
                                       index=structure_names, 
                                       columns=structure_names)
    matrix_file = WORK_DIR / 'similarity_matrix.csv'
    similarity_matrix_df.to_csv(matrix_file)
    print(f"✓ 相似性矩阵保存: {matrix_file}")
    
    return {
        'similarity_matrix': similarity_matrix_df,
        'rmsd_matrix': pd.DataFrame(rmsd_matrix, index=structure_names, columns=structure_names),
        'structure_names': structure_names,
        'similarity_results': results_df
    }

# 运行相似性分析
if quality_results and len(quality_results) >= 2:
    similarity_data = batch_similarity_analysis(quality_results)
else:
    similarity_data = None
    print("⚠️ 无法进行相似性分析: 需要至少2个有效结构")

## 5. 结构聚类分析

In [ ]:
def perform_clustering_analysis(similarity_data):
    """执行聚类分析"""
    
    if not similarity_data:
        print("没有相似性数据")
        return None
    
    try:
        from sklearn.cluster import AgglomerativeClustering
        from sklearn.decomposition import PCA
        from scipy.cluster.hierarchy import dendrogram, linkage
        from scipy.spatial.distance import squareform
        
        print("=== 结构聚类分析 ===")
        
        # 将相似性转换为距离 (1 - 相似性)
        similarity_matrix = similarity_data['similarity_matrix'].values
        distance_matrix = 1 - similarity_matrix
        
        # 层次聚类
        n_samples = len(similarity_data['structure_names'])
        
        if n_samples < 3:
            print(f"⚠️ 样本数过少 ({n_samples})，无法进行有意义的聚类")
            return None
        
        # 不同聚类数量的尝试
        best_n_clusters = 2
        best_score = -1
        
        for n_clusters in range(2, min(n_samples, 8)):  # 最多7个聚类
            clustering = AgglomerativeClustering(
                n_clusters=n_clusters,
                metric='precomputed',
                linkage='average'
            )
            
            labels = clustering.fit_predict(distance_matrix)
            
            # 简单的聚类质量评估 (轮廓系数需要修改用于预计算距离)
            # 这里使用聚类内平均相似性作为指标
            cluster_scores = []
            
            for cluster_id in range(n_clusters):
                cluster_indices = [i for i, label in enumerate(labels) if label == cluster_id]
                
                if len(cluster_indices) > 1:
                    cluster_similarities = []
                    for i in cluster_indices:
                        for j in cluster_indices:
                            if i != j:
                                cluster_similarities.append(similarity_matrix[i, j])
                    
                    cluster_scores.append(np.mean(cluster_similarities))
            
            if cluster_scores:
                avg_score = np.mean(cluster_scores)
                if avg_score > best_score:
                    best_score = avg_score
                    best_n_clusters = n_clusters
                    best_labels = labels
        
        print(f"最佳聚类数: {best_n_clusters}")
        print(f"聚类质量评分: {best_score:.3f}")
        
        # 显示聚类结果
        cluster_info = {}
        for i, label in enumerate(best_labels):
            if label not in cluster_info:
                cluster_info[label] = []
            cluster_info[label].append(similarity_data['structure_names'][i])
        
        print(f"\n聚类结果:")
        for cluster_id, members in cluster_info.items():
            print(f"  聚类 {cluster_id + 1}: {len(members)} 个结构")
            for member in members:
                print(f"    - {member}")
        
        # PCA降维可视化
        pca = PCA(n_components=2, random_state=42)
        
        # 使用距离矩阵进行MDS-like降维
        # 这里简单使用PCA对相似性矩阵进行处理
        pca_result = pca.fit_transform(similarity_matrix)
        
        # 保存聚类结果
        clustering_results = pd.DataFrame({
            'structure_name': similarity_data['structure_names'],
            'cluster': best_labels,
            'pca1': pca_result[:, 0],
            'pca2': pca_result[:, 1]
        })
        
        clustering_file = WORK_DIR / 'structure_clustering.csv'
        clustering_results.to_csv(clustering_file, index=False)
        print(f"\n✓ 聚类结果保存: {clustering_file}")
        
        return {
            'labels': best_labels,
            'cluster_info': cluster_info,
            'pca_result': pca_result,
            'n_clusters': best_n_clusters,
            'score': best_score
        }
    
    except ImportError:
        print("⚠️ 需要安装 scikit-learn 进行聚类分析")
        print("运行: pip install scikit-learn")
        return None
    
    except Exception as e:
        print(f"❌ 聚类分析失败: {e}")
        return None

# 执行聚类分析
if similarity_data:
    clustering_results = perform_clustering_analysis(similarity_data)
else:
    clustering_results = None
    print("⚠️ 无法进行聚类分析: 缺少相似性数据")

## 6. 结果可视化

In [ ]:
def visualize_analysis_results(quality_results, similarity_data, clustering_results):
    """可视化分析结果"""
    
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        
        # 设置图形样式
        plt.style.use('default')
        fig = plt.figure(figsize=(20, 16))
        
        # 1. 结构质量分布
        ax1 = plt.subplot(2, 3, 1)
        successful_results = [r for r in quality_results if r['status'] == 'success']
        
        if successful_results:
            residues = [r['num_residues'] for r in successful_results]
            ax1.hist(residues, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
            ax1.set_xlabel('残基数量')
            ax1.set_ylabel('结构数量')
            ax1.set_title('结构大小分布')
            ax1.axvline(np.mean(residues), color='red', linestyle='--', 
                       label=f'平均值: {np.mean(residues):.0f}')
            ax1.legend()
        
        # 2. 质量状态统计
        ax2 = plt.subplot(2, 3, 2)
        status_counts = {}
        for result in quality_results:
            status = result['status']
            status_counts[status] = status_counts.get(status, 0) + 1
        
        if status_counts:
            statuses = list(status_counts.keys())
            counts = list(status_counts.values())
            colors = ['green', 'orange', 'red', 'gray']
            
            bars = ax2.bar(statuses, counts, color=colors[:len(statuses)], alpha=0.7)
            ax2.set_xlabel('状态')
            ax2.set_ylabel('数量')
            ax2.set_title('结构质量状态分布')
            
            # 添加数值标签
            for bar, count in zip(bars, counts):
                ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                        str(count), ha='center', va='bottom')
        
        # 3. 相似性矩阵热图
        if similarity_data:
            ax3 = plt.subplot(2, 3, 3)
            similarity_matrix = similarity_data['similarity_matrix']
            
            sns.heatmap(similarity_matrix, 
                       annot=False, 
                       cmap='viridis', 
                       square=True,
                       ax=ax3,
                       cbar_kws={'label': '相似性分数'})
            
            ax3.set_title('结构相似性矩阵')
            
            # 简化标签
            if len(similarity_matrix) <= 20:  # 只在结构数较少时显示标签
                labels = [name[:10] + '...' if len(name) > 10 else name 
                         for name in similarity_data['structure_names']]
                ax3.set_xticklabels(labels, rotation=45, ha='right')
                ax3.set_yticklabels(labels, rotation=0)
        
        # 4. PCA散点图
        if clustering_results:
            ax4 = plt.subplot(2, 3, 4)
            pca_result = clustering_results['pca_result']
            labels = clustering_results['labels']
            
            # 为不同聚类使用不同颜色
            unique_labels = np.unique(labels)
            colors = plt.cm.Set3(np.linspace(0, 1, len(unique_labels)))
            
            for i, (label, color) in enumerate(zip(unique_labels, colors)):
                mask = labels == label
                ax4.scatter(pca_result[mask, 0], pca_result[mask, 1],
                           c=[color], label=f'聚类 {label + 1}',
                           s=50, alpha=0.7)
            
            ax4.set_xlabel('PC1')
            ax4.set_ylabel('PC2')
            ax4.set_title('结构PCA可视化')
            ax4.legend()
            ax4.grid(True, alpha=0.3)
        
        # 5. RMSD分布
        if similarity_data:
            ax5 = plt.subplot(2, 3, 5)
            rmsd_values = []
            
            for i in range(len(similarity_data['structure_names'])):
                for j in range(i + 1, len(similarity_data['structure_names'])):
                    rmsd = similarity_data['rmsd_matrix'].iloc[i, j]
                    if not pd.isna(rmsd):
                        rmsd_values.append(rmsd)
            
            if rmsd_values:
                ax5.hist(rmsd_values, bins=20, alpha=0.7, color='lightcoral', edgecolor='black')
                ax5.set_xlabel('RMSD (Å)')
                ax5.set_ylabel('结构对数量')
                ax5.set_title('RMSD分布')
                ax5.axvline(np.mean(rmsd_values), color='darkred', linestyle='--',
                           label=f'平均值: {np.mean(rmsd_values):.1f} Å')
                ax5.legend()
        
        # 6. 聚类大小分布
        if clustering_results:
            ax6 = plt.subplot(2, 3, 6)
            cluster_info = clustering_results['cluster_info']
            
            cluster_sizes = [len(members) for members in cluster_info.values()]
            
            if cluster_sizes:
                bars = ax6.bar(range(len(cluster_sizes)), sorted(cluster_sizes, reverse=True),
                              color='lightgreen', alpha=0.7)
                ax6.set_xlabel('聚类编号')
                ax6.set_ylabel('结构数量')
                ax6.set_title('聚类大小分布')
                
                # 添加数值标签
                for bar, size in zip(bars, sorted(cluster_sizes, reverse=True)):
                    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                            str(size), ha='center', va='bottom')
        
        plt.tight_layout()
        
        # 保存图形
        plot_file = WORK_DIR / 'batch_analysis_results.png'
        plt.savefig(plot_file, dpi=300, bbox_inches='tight')
        print(f"✓ 分析可视化结果保存: {plot_file}")
        
        plt.show()
        
    except ImportError:
        print("⚠️ 需要安装 matplotlib 和 seaborn 进行可视化")
        print("运行: pip install matplotlib seaborn")
    
    except Exception as e:
        print(f"❌ 可视化失败: {e}")

# 运行可视化
if quality_results:
    visualize_analysis_results(quality_results, similarity_data, clustering_results)
else:
    print("⚠️ 没有分析结果可供可视化")

## 7. 综合分析报告

In [ ]:
def generate_comprehensive_report(quality_results, similarity_data, clustering_results):
    """生成综合分析报告"""
    
    report_file = WORK_DIR / 'analysis_report.txt'
    
    with open(report_file, 'w', encoding='utf-8') as f:
        f.write("蛋白质结构批量分析报告\n")
        f.write("=" * 50 + "\n\n")
        
        # 基本信息
        f.write(f"分析时间: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"工作目录: {WORK_DIR}\n\n")
        
        # 质量评估结果
        f.write("1. 结构质量评估\n")
        f.write("-" * 30 + "\n")
        
        status_counts = {}
        for result in quality_results:
            status = result['status']
            status_counts[status] = status_counts.get(status, 0) + 1
        
        f.write(f"总结构数: {len(quality_results)}\n")
        for status, count in status_counts.items():
            percentage = (count / len(quality_results)) * 100
            f.write(f"  {status}: {count} ({percentage:.1f}%)\n")
        
        # 成功结构的统计
        successful = [r for r in quality_results if r['status'] == 'success']
        if successful:
            residues = [r['num_residues'] for r in successful]
            atoms = [r['num_atoms'] for r in successful]
            
            f.write(f"\n成功结构统计:")
            f.write(f"  平均残基数: {np.mean(residues):.1f}\n")
            f.write(f"  残基数范围: {min(residues)} - {max(residues)}\n")
            f.write(f"  平均原子数: {np.mean(atoms):.1f}\n")
            f.write(f"  原子数范围: {min(atoms)} - {max(atoms)}\n")
        
        # 相似性分析结果
        if similarity_data:
            f.write(f"\n2. 结构相似性分析\n")
            f.write("-" * 30 + "\n")
            
            n_structures = len(similarity_data['structure_names'])
            f.write(f"分析结构数: {n_structures}\n")
            f.write(f"结构对总数: {n_structures * (n_structures - 1) // 2}\n")
            
            # 相似性统计
            similarities = []
            rmsds = []
            for i in range(n_structures):
                for j in range(i + 1, n_structures):
                    sim = similarity_data['similarity_matrix'].iloc[i, j]
                    rmsd = similarity_data['rmsd_matrix'].iloc[i, j]
                    similarities.append(sim)
                    if not pd.isna(rmsd):
                        rmsds.append(rmsd)
            
            if similarities:
                f.write(f"平均相似性: {np.mean(similarities):.3f}\n")
                f.write(f"相似性范围: {np.min(similarities):.3f} - {np.max(similarities):.3f}\n")
            
            if rmsds:
                f.write(f"平均RMSD: {np.mean(rmsds):.2f} Å\n")
                f.write(f"RMSD范围: {np.min(rmsds):.2f} - {np.max(rmsds):.2f} Å\n")
        
        # 聚类结果
        if clustering_results:
            f.write(f"\n3. 结构聚类分析\n")
            f.write("-" * 30 + "\n")
            
            f.write(f"聚类数: {clustering_results['n_clusters']}\n")
            f.write(f"聚类质量评分: {clustering_results['score']:.3f}\n\n")
            
            cluster_info = clustering_results['cluster_info']
            for cluster_id, members in cluster_info.items():
                f.write(f"聚类 {cluster_id + 1}: {len(members)} 个结构\n")
                for member in members:
                    f.write(f"  - {member}\n")
                f.write("\n")
        
        # 结果文件位置
        f.write(f"\n4. 结果文件\n")
        f.write("-" * 30 + "\n")
        f.write(f"质量评估结果: {WORK_DIR / 'structure_quality.csv'}\n")
        
        if similarity_data:
            f.write(f"相似性结果: {WORK_DIR / 'structure_similarity.csv'}\n")
            f.write(f"相似性矩阵: {WORK_DIR / 'similarity_matrix.csv'}\n")
        
        if clustering_results:
            f.write(f"聚类结果: {WORK_DIR / 'structure_clustering.csv'}\n")
        
        f.write(f"可视化图表: {WORK_DIR / 'batch_analysis_results.png'}\n")
        
        # 建议和下一步
        f.write(f"\n5. 建议\n")
        f.write("-" * 30 + "\n")
        
        if clustering_results:
            f.write("基于聚类结果，建议对每个聚类进行代表性结构分析:")
            cluster_info = clustering_results['cluster_info']
            for cluster_id, members in cluster_info.items():
                if len(members) > 0:
                    f.write(f"  - 聚类 {cluster_id + 1}: 推荐分析 {members[0]}\n")
        
        if successful:
            # 找出最大和最小的结构
            largest = max(successful, key=lambda x: x['num_residues'])
            smallest = min(successful, key=lambda x: x['num_residues'])
            
            f.write(f"\n特殊结构推荐:")
            f.write(f"  - 最大结构: {largest['protein_name']} ({largest['num_residues']} 残基)\n")
            f.write(f"  - 最小结构: {smallest['protein_name']} ({smallest['num_residues']} 残基)\n")
        
        f.write(f"\n可使用以下工具进行进一步分析:")
        f.write(f"  - 01_protein_structure_prediction.ipynb: 结构预测\n")
        f.write(f"  - 02_pocket_detection_p2rank.ipynb: 口袋检测\n")
        f.write(f"  - 12_structure_alignment_dali.ipynb: 结构比对\n")
    
    print(f"✓ 综合分析报告生成: {report_file}")
    
    # 显示报告摘要
    print("\n=== 分析报告摘要 ===")
    print(f"总结构数: {len(quality_results)}")
    
    successful_count = len([r for r in quality_results if r['status'] == 'success'])
    print(f"成功结构: {successful_count}")
    
    if similarity_data:
        print(f"相似性分析: {len(similarity_data['structure_names'])} 个结构")
    
    if clustering_results:
        print(f"聚类分析: {clustering_results['n_clusters']} 个聚类")
    
    print(f"详细报告见: {report_file}")

# 生成报告
if quality_results:
    generate_comprehensive_report(quality_results, similarity_data, clustering_results)
else:
    print("⚠️ 没有分析结果，无法生成报告")

## 8. 下一步操作

完成批量结构分析后，您可以：

1. **详细分析**:
   - 选择代表性结构进行口袋检测 (`02_pocket_detection_p2rank.ipynb`)
   - 对关键结构进行分子对接 (`03_ligand_docking_vina.ipynb`)
   - 使用DALI进行更精确的结构比对 (`12_structure_alignment_dali.ipynb`)

2. **聚类分析**:
   - 对每个聚类的代表结构进行功能分析
   - 研究聚类间的功能差异
   - 分析聚类与蛋白质功能的关系

3. **质量控制**:
   - 基于质量评估结果过滤低质量结构
   - 重新预测或优化质量较差的结构
   - 建立结构质量标准

4. **功能注释**:
   - 结合序列分析工具进行功能预测
   - 比较结构相似性与序列相似性的关系
   - 研究结构保守性与功能的关系

**结果文件位置:**
- 质量评估: `{WORK_DIR}/structure_quality.csv`
- 相似性矩阵: `{WORK_DIR}/similarity_matrix.csv`
- 聚类结果: `{WORK_DIR}/structure_clustering.csv`
- 可视化图表: `{WORK_DIR}/batch_analysis_results.png`
- 综合报告: `{WORK_DIR}/analysis_report.txt`

**注意:**
- 大规模结构分析需要较长的计算时间
- 建议先在小规模数据集上测试参数
- 结构相似性分析对内存要求较高
- 结果解释需要结合生物学背景知识